In [22]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

PROJECT_ROOT = r"C:\Users\Mayank Joshi\Downloads\Marketing_Channel_Project"
DB_PATH      = os.path.join(PROJECT_ROOT, "olist_analytics.db")
DATA_PATH    = os.path.join(PROJECT_ROOT, "data")
OUTPUT_PATH  = os.path.join(DATA_PATH, "sql_analysis")
os.makedirs(OUTPUT_PATH, exist_ok=True)

con = duckdb.connect(DB_PATH)

print(f"Connected to: {DB_PATH}")
print(f"Tables available:")
tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'main' ORDER BY table_name"
).fetchall()
for t in tables:
    print(f"  - {t[0]}")

Connected to: C:\Users\Mayank Joshi\Downloads\Marketing_Channel_Project\olist_analytics.db
Tables available:
  - activated_sellers
  - attribution_comparison
  - category_translation
  - channel_funnel
  - closed_deals
  - cohort_monthly
  - customers
  - geolocation
  - high_value_cluster
  - ltv_by_channel
  - ltv_by_segment
  - master
  - mql
  - order_items
  - order_payments
  - order_reviews
  - orders
  - products
  - rfm_scores
  - sellers
  - sellers_segmented


In [23]:
query = """
WITH funnel AS (
    SELECT
        COUNT(*)                                        AS total_leads,
        SUM(CASE WHEN first_contact_date IS NOT NULL
                 THEN 1 ELSE 0 END)                    AS leads_with_date,
        SUM(converted)                                  AS closed_deals,
        SUM(CASE WHEN total_orders > 0
                 THEN 1 ELSE 0 END)                    AS made_first_order,
        SUM(CASE WHEN total_orders >= 3
                 THEN 1 ELSE 0 END)                    AS repeat_sellers
    FROM master
)
SELECT
    'Leads generated'           AS stage, total_leads      AS count,
    100.0                       AS pct_of_total
FROM funnel
UNION ALL
SELECT
    'Leads with contact date',  leads_with_date,
    ROUND(leads_with_date * 100.0 / total_leads, 2)
FROM funnel
UNION ALL
SELECT
    'Closed deals',             closed_deals,
    ROUND(closed_deals * 100.0 / total_leads, 2)
FROM funnel
UNION ALL
SELECT
    'Made first order',         made_first_order,
    ROUND(made_first_order * 100.0 / total_leads, 2)
FROM funnel
UNION ALL
SELECT
    'Repeat seller (3+ orders)',repeat_sellers,
    ROUND(repeat_sellers * 100.0 / total_leads, 2)
FROM funnel
ORDER BY count DESC
"""

funnel_df = con.execute(query).df()
print("FUNNEL DROP-OFF ANALYSIS")
print("=" * 50)
print(funnel_df.to_string(index=False))

FUNNEL DROP-OFF ANALYSIS
                    stage   count  pct_of_total
          Leads generated 8000.00        100.00
  Leads with contact date 3058.00         38.23
             Closed deals  842.00         10.53
         Made first order  380.00          4.75
Repeat seller (3+ orders)  252.00          3.15


In [24]:
query = """
SELECT
    origin                                              AS channel,
    COUNT(*)                                            AS total_leads,
    SUM(converted)                                      AS conversions,
    ROUND(SUM(converted) * 100.0 / COUNT(*), 2)        AS cvr_pct,
    SUM(CASE WHEN total_orders > 0 THEN 1 ELSE 0 END)  AS activated_sellers,
    ROUND(
        SUM(CASE WHEN total_orders > 0 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(converted), 0), 1
    )                                                   AS activation_rate_pct,
    ROUND(SUM(total_revenue), 2)                        AS total_revenue,
    ROUND(SUM(total_revenue) / COUNT(*), 2)             AS revenue_per_lead
FROM master
GROUP BY origin
ORDER BY cvr_pct DESC
"""

cvr_df = con.execute(query).df()
print("CONVERSION RATE BY CHANNEL")
print("=" * 85)
print(cvr_df.to_string(index=False))

CONVERSION RATE BY CHANNEL
          channel  total_leads  conversions  cvr_pct  activated_sellers  activation_rate_pct  total_revenue  revenue_per_lead
          unknown         1159       193.00    16.65              85.00                44.00      214985.30            185.49
      paid_search         1586       195.00    12.30             101.00                51.80      155277.05             97.90
   organic_search         2296       271.00    11.80             113.00                41.70      207023.45             90.17
   direct_traffic          499        56.00    11.22              31.00                55.40       21903.90             43.90
         referral          284        24.00     8.45               9.00                37.50       17887.15             62.98
           social         1350        75.00     5.56              31.00                41.30       43477.99             32.21
          display          118         6.00     5.08               2.00                33.3

In [26]:
query = """
SELECT
    STRFTIME('%Y-%m', TRY_CAST(first_contact_date AS DATE)) AS cohort_month,
    COUNT(*)                                            AS total_leads,
    SUM(converted)                                      AS conversions,
    ROUND(SUM(converted) * 100.0 / COUNT(*), 2)        AS cvr_pct,
    ROUND(SUM(total_revenue), 2)                        AS total_revenue,
    ROUND(SUM(total_revenue) / COUNT(*), 2)             AS revenue_per_lead
FROM master
WHERE first_contact_date IS NOT NULL
GROUP BY cohort_month
ORDER BY cohort_month
"""

cohort_df = con.execute(query).df()
print("MONTHLY COHORT ANALYSIS")
print("=" * 70)
print(cohort_df.to_string(index=False))

MONTHLY COHORT ANALYSIS
cohort_month  total_leads  conversions  cvr_pct  total_revenue  revenue_per_lead
     2017-01           76         3.00     3.95        2368.80             31.17
     2017-02           39         1.00     2.56         312.99              8.03
     2017-03           49         0.00     0.00           0.00              0.00
     2017-04           59         1.00     1.69           0.00              0.00
     2017-05           52         1.00     1.92           0.00              0.00
     2017-06           76         0.00     0.00           0.00              0.00
     2017-07           66         3.00     4.55         604.80              9.16
     2017-08           42         2.00     4.76           0.00              0.00
     2017-09           49         1.00     2.04         573.50             11.70
     2017-10           79         4.00     5.06          89.90              1.14
     2017-11           51         2.00     3.92           0.00              0.00
    

In [27]:
query = """
WITH seller_revenue AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id)    AS total_orders,
        ROUND(SUM(price), 2)        AS total_revenue,
        ROUND(AVG(price), 2)        AS avg_item_price
    FROM order_items
    GROUP BY seller_id
)
SELECT
    mql.origin                                          AS channel,
    COUNT(mql.mql_id)                                  AS total_leads,
    COUNT(cd.mql_id)                                   AS conversions,
    ROUND(COUNT(cd.mql_id) * 100.0
          / COUNT(mql.mql_id), 2)                      AS cvr_pct,
    COUNT(sr.seller_id)                                AS activated_sellers,
    ROUND(SUM(COALESCE(sr.total_revenue, 0)), 2)       AS total_revenue,
    ROUND(AVG(COALESCE(sr.total_revenue, 0)), 2)       AS avg_ltv,
    ROUND(SUM(COALESCE(sr.total_revenue, 0))
          / NULLIF(COUNT(mql.mql_id), 0), 2)           AS revenue_per_lead
FROM mql
LEFT JOIN closed_deals cd   ON mql.mql_id    = cd.mql_id
LEFT JOIN seller_revenue sr ON cd.seller_id  = sr.seller_id
GROUP BY mql.origin
ORDER BY total_revenue DESC
"""

attribution_sql = con.execute(query).df()
print("FULL ATTRIBUTION CHAIN — MQL TO REVENUE")
print("=" * 90)
print(attribution_sql.to_string(index=False))

attribution_sql.to_csv(
    os.path.join(OUTPUT_PATH, 'attribution_chain.csv'), index=False
)
print("\nattribution_chain.csv saved")

FULL ATTRIBUTION CHAIN — MQL TO REVENUE
          channel  total_leads  conversions  cvr_pct  activated_sellers  total_revenue  avg_ltv  revenue_per_lead
          unknown         1099          179    16.29                 81      213742.70   194.49            194.49
   organic_search         2296          271    11.80                113      207023.45    90.17             90.17
      paid_search         1586          195    12.30                101      155277.05    97.90             97.90
           social         1350           75     5.56                 31       43477.99    32.21             32.21
   direct_traffic          499           56    11.22                 31       21903.90    43.90             43.90
         referral          284           24     8.45                  9       17887.15    62.98             62.98
            email          493           15     3.04                  6        8484.99    17.21             17.21
            other          150            4     

In [28]:
query = """
WITH seller_revenue AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(price), 2)     AS total_revenue
    FROM order_items
    GROUP BY seller_id
)
SELECT
    d.business_segment,
    COUNT(d.seller_id)                              AS total_sellers,
    COUNT(sr.seller_id)                             AS activated_sellers,
    ROUND(AVG(COALESCE(sr.total_revenue, 0)), 2)   AS avg_ltv,
    ROUND(MEDIAN(COALESCE(sr.total_revenue, 0)), 2) AS median_ltv,
    ROUND(SUM(COALESCE(sr.total_revenue, 0)), 2)   AS total_revenue,
    ROUND(AVG(COALESCE(sr.total_orders, 0)), 1)    AS avg_orders
FROM closed_deals d
LEFT JOIN seller_revenue sr ON d.seller_id = sr.seller_id
WHERE d.business_segment IS NOT NULL
GROUP BY d.business_segment
HAVING activated_sellers >= 2
ORDER BY avg_ltv DESC
LIMIT 15
"""

ltv_seg_sql = con.execute(query).df()
print("LTV BY BUSINESS SEGMENT")
print("=" * 80)
print(ltv_seg_sql.to_string(index=False))

LTV BY BUSINESS SEGMENT
       business_segment  total_sellers  activated_sellers  avg_ltv  median_ltv  total_revenue  avg_orders
                watches              8                  3 14659.86        0.00      117278.87       72.50
       small_appliances             12                  7  3972.91      282.94       47674.87        5.70
        home_appliances              7                  5  3748.74      573.50       26241.17       19.30
                   baby             10                  6  1632.10      194.90       16320.96        4.10
                    pet             30                 17  1349.96      104.85       40498.83        8.20
          health_beauty             93                 45   976.73        0.00       90835.82        7.60
       air_conditioning              3                  2   883.33      560.00        2650.00        1.30
audio_video_electronics             64                 31   785.40        0.00       50265.34        3.90
                  book

In [29]:
query = """
WITH seller_revenue AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id)                AS total_orders,
        ROUND(SUM(price), 2)                    AS total_revenue,
        ROUND(AVG(price), 2)                    AS avg_item_price,
        COUNT(DISTINCT product_id)              AS unique_products
    FROM order_items
    GROUP BY seller_id
),
ranked AS (
    SELECT
        sr.seller_id,
        sr.total_revenue,
        sr.total_orders,
        sr.avg_item_price,
        sr.unique_products,
        d.business_segment,
        d.business_type,
        d.lead_type,
        m.origin,
        ROW_NUMBER() OVER (ORDER BY sr.total_revenue DESC) AS revenue_rank,
        ROUND(
            SUM(sr.total_revenue) OVER (ORDER BY sr.total_revenue DESC)
            / SUM(sr.total_revenue) OVER () * 100, 2
        )                                       AS cumulative_revenue_pct,
        ROUND(
            sr.total_revenue * 100.0
            / SUM(sr.total_revenue) OVER (), 2
        )                                       AS revenue_share_pct
    FROM seller_revenue sr
    LEFT JOIN closed_deals d ON sr.seller_id = d.seller_id
    LEFT JOIN mql          m ON d.mql_id     = m.mql_id
)
SELECT
    revenue_rank,
    seller_id,
    origin                  AS channel,
    business_segment,
    business_type,
    lead_type,
    total_orders,
    total_revenue,
    revenue_share_pct,
    cumulative_revenue_pct
FROM ranked
WHERE revenue_rank <= 20
ORDER BY revenue_rank
"""

top_sellers = con.execute(query).df()
print("TOP 20 SELLERS BY REVENUE — WINDOW FUNCTION RANKING")
print("=" * 90)
print(top_sellers.to_string(index=False))

TOP 20 SELLERS BY REVENUE — WINDOW FUNCTION RANKING
 revenue_rank                        seller_id channel business_segment business_type  lead_type  total_orders  total_revenue  revenue_share_pct  cumulative_revenue_pct
            1 4869f7a5dfa277a7dca6462dcf3b52b2    None             None          None       None          1132      229472.63               1.69                    1.69
            2 53243585a1d6dc2643021fd1853d8905    None             None          None       None           358      222776.05               1.64                    3.33
            3 4a3ca9315b744ce9f8e9374361493884    None             None          None       None          1806      200472.92               1.47                    4.80
            4 fa1c13f2614d7b5c4749cbc52fecda94    None             None          None       None           585      194042.03               1.43                    6.23
            5 7c67e1448b00f6e969d365cea6b010ab    None             None          None       None       

In [30]:
query = """
SELECT
    m.landing_page_id,
    COUNT(m.mql_id)                                       AS total_leads,
    SUM(CASE WHEN cd.mql_id IS NOT NULL THEN 1 ELSE 0 END) AS conversions,
    ROUND(
        SUM(CASE WHEN cd.mql_id IS NOT NULL THEN 1 ELSE 0 END) * 100.0
        / COUNT(m.mql_id), 2
    )                                                     AS cvr_pct,
    COUNT(m.mql_id) - SUM(CASE WHEN cd.mql_id IS NOT NULL
                               THEN 1 ELSE 0 END)        AS non_conversions
FROM mql m
LEFT JOIN closed_deals cd ON m.mql_id = cd.mql_id
GROUP BY m.landing_page_id
HAVING total_leads >= 30
ORDER BY cvr_pct DESC
"""

landing_sql = con.execute(query).df()
print("LANDING PAGE EXPERIMENT — NATURAL A/B TEST")
print("=" * 65)
print(landing_sql.to_string(index=False))

from scipy import stats
contingency    = landing_sql[['conversions', 'non_conversions']].values
chi2, p_value, dof, _ = stats.chi2_contingency(contingency)
print(f"\nChi-square test:")
print(f"  Chi2 statistic     : {chi2:.4f}")
print(f"  Degrees of freedom : {dof}")
print(f"  p-value            : {p_value:.6f}")
print(f"  Result: {'SIGNIFICANT' if p_value < 0.05 else 'NOT SIGNIFICANT'} at 95% confidence")

LANDING PAGE EXPERIMENT — NATURAL A/B TEST
                 landing_page_id  total_leads  conversions  cvr_pct  non_conversions
30077c17f2ec5010a82e37ad8925b95f           48        10.00    20.83            38.00
40dec9f3d5259a3d2dbcdab2114fae47          330        67.00    20.30           263.00
22c29808c4f815213303f8933030604c          883       174.00    19.71           709.00
b76ef37428e6799c421989521c0e5077          912       171.00    18.75           741.00
d83b0d0e48c8447d1d5507a44027a955           34         6.00    17.65            28.00
7fa6214d82e911d070f51ef79381b956           68        11.00    16.18            57.00
ce1a65abd0973638f1c887a6efcfa82d          394        59.00    14.97           335.00
b48ec5f3b04e9068441002a19df93c6c           51         7.00    13.73            44.00
35c9b150ab36fe584c1f24fd458c453a           59         8.00    13.56            51.00
241f79c7a8fe0270f4fb79fcbbcd17ad          109        14.00    12.84            95.00
1ceb590cd1e00c7ee95220

In [31]:
query = """
WITH seller_metrics AS (
    SELECT
        d.seller_id,
        d.business_segment,
        d.lead_type,
        d.business_type,
        m.origin,
        DATEDIFF('day', TRY_CAST(d.won_date AS DATE),
                  DATE '2018-08-08')                    AS recency_days,
        COUNT(DISTINCT oi.order_id)                     AS frequency,
        ROUND(SUM(oi.price), 2)                         AS monetary
    FROM closed_deals d
    LEFT JOIN mql          m  ON d.mql_id    = m.mql_id
    LEFT JOIN order_items  oi ON d.seller_id = oi.seller_id
    WHERE oi.seller_id IS NOT NULL
    GROUP BY d.seller_id, d.business_segment, d.lead_type,
             d.business_type, m.origin, d.won_date
),
rfm_scored AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY recency_days DESC)  AS r_score,
        NTILE(4) OVER (ORDER BY frequency ASC)      AS f_score,
        NTILE(4) OVER (ORDER BY monetary ASC)       AS m_score
    FROM seller_metrics
),
rfm_segmented AS (
    SELECT *,
        r_score + f_score + m_score                 AS rfm_total,
        CASE
            WHEN r_score + f_score + m_score >= 10 THEN 'champions'
            WHEN r_score + f_score + m_score >= 8  THEN 'loyal'
            WHEN r_score + f_score + m_score >= 6  THEN 'promising'
            WHEN r_score + f_score + m_score >= 4  THEN 'at_risk'
            ELSE 'dormant'
        END                                         AS rfm_segment
    FROM rfm_scored
)
SELECT
    rfm_segment,
    COUNT(*)                        AS sellers,
    ROUND(AVG(monetary), 2)         AS avg_revenue,
    ROUND(AVG(frequency), 1)        AS avg_orders,
    ROUND(AVG(recency_days), 0)     AS avg_recency_days
FROM rfm_segmented
GROUP BY rfm_segment
ORDER BY avg_revenue DESC
"""

rfm_sql = con.execute(query).df()
print("RFM SEGMENTATION — SQL NTILE APPROACH")
print("=" * 65)
print(rfm_sql.to_string(index=False))

RFM SEGMENTATION — SQL NTILE APPROACH
rfm_segment  sellers  avg_revenue  avg_orders  avg_recency_days
  champions       72      3840.59       28.70            101.00
      loyal      112      3011.92       16.80            129.00
  promising      120       429.03        3.70            123.00
    at_risk       62       163.12        1.90            148.00
    dormant       14        99.73        1.00            185.00


In [32]:
query = """
SELECT
    payment_type,
    COUNT(DISTINCT order_id)              AS total_orders,
    ROUND(AVG(payment_installments), 1)   AS avg_installments,
    ROUND(AVG(payment_value), 2)          AS avg_order_value,
    ROUND(SUM(payment_value), 2)          AS total_revenue,
    ROUND(
        COUNT(DISTINCT order_id) * 100.0
        / SUM(COUNT(DISTINCT order_id)) OVER (), 1
    )                                     AS pct_of_orders
FROM order_payments
GROUP BY payment_type
ORDER BY total_revenue DESC
"""

payments_sql = con.execute(query).df()
print("PAYMENT METHOD ANALYSIS")
print("=" * 70)
print(payments_sql.to_string(index=False))

PAYMENT METHOD ANALYSIS
payment_type  total_orders  avg_installments  avg_order_value  total_revenue  pct_of_orders
 credit_card         76505              3.50           163.32    12542084.19          75.20
      boleto         19784              1.00           145.03     2869361.27          19.50
     voucher          3866              1.00            65.70      379436.87           3.80
  debit_card          1528              1.00           142.57      217989.79           1.50
 not_defined             3              1.00             0.00           0.00           0.00


In [33]:
cvr_df.to_csv(os.path.join(OUTPUT_PATH, 'cvr_by_channel.csv'), index=False)
cohort_df.to_csv(os.path.join(OUTPUT_PATH, 'cohort_monthly.csv'), index=False)
attribution_sql.to_csv(os.path.join(OUTPUT_PATH, 'attribution_chain.csv'), index=False)
ltv_seg_sql.to_csv(os.path.join(OUTPUT_PATH, 'ltv_by_segment.csv'), index=False)
top_sellers.to_csv(os.path.join(OUTPUT_PATH, 'top_sellers_ranked.csv'), index=False)
landing_sql.to_csv(os.path.join(OUTPUT_PATH, 'landing_page_experiment.csv'), index=False)
rfm_sql.to_csv(os.path.join(OUTPUT_PATH, 'rfm_segments_sql.csv'), index=False)
payments_sql.to_csv(os.path.join(OUTPUT_PATH, 'payment_analysis.csv'), index=False)

con.close()

print("SQL ANALYSIS COMPLETE")
print("=" * 50)
print(f"\n  Source          : olist_analytics.db")
print(f"  Tables queried  : mql, closed_deals, order_items,")
print(f"                    order_payments, master")
print(f"  Queries written : 10")
print(f"  Techniques used :")
print(f"    CTEs, LEFT JOINs across 4 tables")
print(f"    Window functions — ROW_NUMBER(), SUM() OVER()")
print(f"    NTILE() for RFM scoring")
print(f"    CASE WHEN, NULLIF, COALESCE")
print(f"    STRFTIME for date truncation")
print(f"    DATEDIFF for recency calculation")
print(f"    HAVING for post-aggregation filtering")
print(f"\n  Outputs saved to: {OUTPUT_PATH}")

SQL ANALYSIS COMPLETE

  Source          : olist_analytics.db
  Tables queried  : mql, closed_deals, order_items,
                    order_payments, master
  Queries written : 10
  Techniques used :
    CTEs, LEFT JOINs across 4 tables
    Window functions — ROW_NUMBER(), SUM() OVER()
    NTILE() for RFM scoring
    CASE WHEN, NULLIF, COALESCE
    STRFTIME for date truncation
    DATEDIFF for recency calculation
    HAVING for post-aggregation filtering

  Outputs saved to: C:\Users\Mayank Joshi\Downloads\Marketing_Channel_Project\data\sql_analysis
